import data

In [35]:
import pandas as pd

# Memuat data dari file CSV
file_path = 'dataset/data_science_job.csv'
df = pd.read_csv(file_path, encoding='latin1')

# Menampilkan 5 baris pertama
print("5 Baris Pertama Data:")
display(df.head())

# Menampilkan informasi struktur data
print("\nInformasi Dataset:")
display(df.info())

5 Baris Pertama Data:


,Company,Job Title,Location,Job Type,Experience level,Salary,Requirment of the company,Facilities
0,SGS,Clinical Data Analyst,"Richardson, TX, United States",Full Time,Entry-level,48K+ *,"Computer Science,Data quality,Genetics,Mathema...",",,,,"
1,Ocorian,AML/CFT & Data Analyst,"Ebène, Mauritius",Full Time,Entry-level,48K+ *,"Agile,Data management,Finance,Security,,",",,,,"
2,Cricut,Machine Learning Engineer,"South Jordan, UT, United States",Full Time,NaN,90K+ *,"Agile,Architecture,AWS,Computer Science,Comput...","Career development,,,,"
3,Bosch Group,Application Developer & Data Analyst,"Nonantola, Italy",Full Time,Entry-level,48K+ *,"Engineering,Industrial,Oracle,Power BI,R,R&D",",,,,"
4,Publicis Groupe,Data Engineer Full time (Public Sector) USA,"Arlington, VA, United States",Full Time,Mid-level,108K+,"AWS,Azure,Computer Science,Consulting,Dataflow...","Flex hours,Flex vacation,Parental leave,Unlimi..."



Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3198 entries, 0 to 3197
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Company                     3197 non-null   object
 1   Job Title                   3197 non-null   object
 2   Location                    3197 non-null   object
 3   Job Type                    3197 non-null   object
 4   Experience level            2962 non-null   object
 5   Salary                      3009 non-null   object
 6   Requirment of the company   3198 non-null   object
 7   Facilities                  3198 non-null   object
dtypes: object(8)
memory usage: 200.0+ KB


None

### Tahap 2: Assessing Data
Pada tahap ini, kita akan memeriksa integritas data, tipe data, dan konsistensi nilai.

In [36]:
# Memeriksa missing values secara detail
print("Jumlah Missing Values per Kolom:")
print(df.isnull().sum())


Jumlah Missing Values per Kolom:
Company                         1
Job Title                       1
Location                        1
Job Type                        1
Experience level              236
Salary                        189
Requirment of the company       0
Facilities                      0
dtype: int64


In [37]:
# Memeriksa data duplikat
print(f"\nJumlah Baris Duplikat: {df.duplicated().sum()}")



Jumlah Baris Duplikat: 202


In [38]:
import pandas as pd

# --- Deteksi Inaccurate Data (Kategorikal) ---
print("--- Deteksi Inaccurate Data ---")

# Standardize column names to prevent KeyError due to hidden whitespaces
df.columns = df.columns.str.strip()

# 1. Deteksi 'Semantic Emptiness' (Data yang secara teknis ada tapi tidak bermakna)
# Contoh: Kolom 'Facilities' atau 'Requirment of the company' yang hanya berisi tanda koma atau spasi
semantic_empty_facilities = df[df['Facilities'].str.match(r'^[\s,]*$', na=False)]
semantic_empty_req = df[df['Requirment of the company'].str.match(r'^[\s,]*$', na=False)]

print(f"Jumlah baris dengan 'Facilities' tidak akurat (hanya koma): {len(semantic_empty_facilities)}")
print(f"Jumlah baris dengan 'Requirment' tidak akurat (hanya koma): {len(semantic_empty_req)}")

# 2. Deteksi Kontradiksi Logis Sederhana
# Contoh: Job Title mengandung 'Senior' tapi Experience level tertulis 'Entry-level'
check_senior_entry = df[(df['Job Title'].str.contains('Senior', case=False, na=False)) &
                        (df['Experience level'] == 'Entry-level')]

print(f"\nPotensi Inakurasi: Judul 'Senior' tapi level 'Entry-level': {len(check_senior_entry)} baris")
if len(check_senior_entry) > 0:
    display(check_senior_entry[['Job Title', 'Experience level']].head())

# 3. Deteksi Item Duplikat dalam Satu Baris (Inkonsistensi String)
# Memeriksa apakah ada requirement yang ditulis berulang kali dalam satu baris
def find_duplicate_items(text):
    if pd.isna(text): return False
    items = [i.strip().lower() for i in text.split(',') if i.strip()]
    return len(items) != len(set(items))

duplicate_reqs = df[df['Requirment of the company'].apply(find_duplicate_items)]
print(f"\nBaris dengan item duplikat di kolom Requirment (Inkonsistensi): {len(duplicate_reqs)}")

# 4. Validasi Nilai Tak Terduga pada 'Job Type'
expected_job_types = ['Full Time', 'Internship', 'Part Time']
invalid_job_types = df[~df['Job Type'].isin(expected_job_types) & df['Job Type'].notna()]
print(f"\nNilai tidak terduga pada kolom 'Job Type': {invalid_job_types['Job Type'].unique()}")

--- Deteksi Inaccurate Data ---
Jumlah baris dengan 'Facilities' tidak akurat (hanya koma): 542
Jumlah baris dengan 'Requirment' tidak akurat (hanya koma): 1

Potensi Inakurasi: Judul 'Senior' tapi level 'Entry-level': 7 baris


,Job Title,Experience level
94,Senior Consultant in Data science,Entry-level
1104,"Senior Data Analyst, International",Entry-level
1298,"Junior, Intermediate or Senior Data Analyst",Entry-level
2431,Senior Insight Analyst,Entry-level
2781,Senior Consultant in Data Science,Entry-level



Baris dengan item duplikat di kolom Requirment (Inkonsistensi): 0

Nilai tidak terduga pada kolom 'Job Type': []


In [39]:
# --- Pemeriksaan Data Tidak Valid (Invalid Data) ---
print("--- Pemeriksaan Data Tidak Valid ---")

# 1. Mencari placeholder string yang menandakan data tidak valid
placeholders = ['unknown', 'none', 'n/a', 'null', '-', '.']
invalid_counts = {}

for col in df.columns:
    counts = df[df[col].astype(str).str.lower().str.strip().isin(placeholders)].shape[0]
    if counts > 0:
        invalid_counts[col] = counts

print("\nJumlah placeholder (seperti 'unknown', '-') per kolom:")
if invalid_counts:
    for col, count in invalid_counts.items():
        print(f"{col}: {count}")
else:
    print("Tidak ditemukan placeholder umum.")

# 2. Cek duplikasi nama Company dengan perbedaan huruf besar/kecil
lowered_companies = df['Company'].dropna().str.lower().str.strip()
print(f"\nJumlah Company unik (case-sensitive): {df['Company'].nunique()}")
print(f"Jumlah Company unik (case-insensitive): {lowered_companies.nunique()}")

# 3. Cek format gaji yang mungkin tidak valid (tidak mengandung angka atau simbol K)
print("\nSampel format 'Salary' yang mungkin bermasalah (tidak ada angka):")
invalid_salary_format = df[~df['Salary'].astype(str).str.contains(r'\d', na=False)]['Salary'].unique()
print(invalid_salary_format)

# 4. Cek baris dengan Facilities yang isinya hanya koma (kosong secara semantik)
only_commas_facilities = df[df['Facilities'].str.match(r'^[\s,]*$', na=False)]
print(f"\nJumlah baris dengan 'Facilities' hanya berisi koma: {len(only_commas_facilities)}")

--- Pemeriksaan Data Tidak Valid ---

Jumlah placeholder (seperti 'unknown', '-') per kolom:
Tidak ditemukan placeholder umum.

Jumlah Company unik (case-sensitive): 1106
Jumlah Company unik (case-insensitive): 1106

Sampel format 'Salary' yang mungkin bermasalah (tidak ada angka):
[nan]

Jumlah baris dengan 'Facilities' hanya berisi koma: 542


In [40]:
# --- Deteksi Outlier dan Inaccurate/Invalid Data (Numerik/Potensi Numerik) ---
print("\n--- Deteksi Outlier dan Inaccurate/Invalid Data (Numerik/Potensi Numerik) ---")

# Fungsi untuk membersihkan dan mengkonversi kolom Salary ke numerik
def clean_salary(salary_str):
    if pd.isna(salary_str):
        return None
    salary_str = str(salary_str).replace(' ', '').replace('$', '').lower()
    if 'k' in salary_str:
        salary_str = salary_str.replace('k', '')
        try:
            return float(salary_str.replace('+', '')) * 1000  # Assume K means thousands
        except ValueError:
            return None
    elif '+' in salary_str:
        try:
            return float(salary_str.replace('+', ''))
        except ValueError:
            return None
    else:
        try:
            return float(salary_str)
        except ValueError:
            return None

# Aplikasikan fungsi pembersihan ke kolom 'Salary'
df['Salary_cleaned_numeric'] = df['Salary'].apply(clean_salary)

print("\nStatistik Deskriptif untuk Kolom 'Salary_cleaned_numeric':")
display(df['Salary_cleaned_numeric'].describe())

# Deteksi Outlier untuk Salary_cleaned_numeric menggunakan metode IQR
if not df['Salary_cleaned_numeric'].isnull().all():
    Q1 = df['Salary_cleaned_numeric'].quantile(0.25)
    Q3 = df['Salary_cleaned_numeric'].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df['Salary_cleaned_numeric'] < lower_bound) | (df['Salary_cleaned_numeric'] > upper_bound)]

    print(f"\nJumlah Outlier pada Kolom 'Salary_cleaned_numeric' (menggunakan IQR): {len(outliers)}")
    if len(outliers) > 0:
        print("Contoh Outlier 'Salary':")
        display(outliers[['Salary', 'Salary_cleaned_numeric']].head())
else:
    print("Kolom 'Salary' tidak dapat dikonversi sepenuhnya ke numerik untuk analisis outlier.")

# Hapus kolom sementara yang dibuat untuk analisis
df.drop(columns=['Salary_cleaned_numeric'], errors='ignore', inplace=True)


--- Deteksi Outlier dan Inaccurate/Invalid Data (Numerik/Potensi Numerik) ---

Statistik Deskriptif untuk Kolom 'Salary_cleaned_numeric':


,Salary_cleaned_numeric
count,583.000000
mean,137864.493997
std,50394.974232
min,40000.000000
25%,104000.000000
50%,136000.000000
75%,166000.000000
max,315000.000000



Jumlah Outlier pada Kolom 'Salary_cleaned_numeric' (menggunakan IQR): 20
Contoh Outlier 'Salary':


,Salary,Salary_cleaned_numeric
596,283K+,283000.0
665,283K+,283000.0
688,295K+,295000.0
720,283K+,283000.0
732,283K+,283000.0


### Ringkasan Assessment (Dataset Data Science Job):
Berdasarkan pemeriksaan mendalam, berikut adalah ringkasan masalah kualitas data yang ditemukan:

1. **Incompleteness (Data Hilang):**
   - **Experience level**: 236 data hilang.
   - **Salary**: 189 data hilang.
   - **Company, Job Title, Location, Job Type**: Masing-masing memiliki 1 data hilang.

2. **Duplication (Data Duplikat):**
   - Ditemukan **202 baris duplikat** yang identik di seluruh dataset.

3. **Inaccuracy & Invalidity (Ketidakakuratan & Data Tidak Valid):**
   - **Semantic Emptiness (Facilities)**: 542 baris kolom `Facilities` hanya berisi koma (`,,,,`), yang berarti secara fungsional kosong.
   - **Kontradiksi Logis**: 7 baris memiliki Judul Pekerjaan mengandung kata 'Senior' namun tingkat pengalaman tertulis 'Entry-level'.
   - **Outliers (Salary)**: Ditemukan **20 data pencilan** pada gaji (hingga 315K) yang perlu divalidasi lebih lanjut.

4. **Inconsistency (Inkonsistensi):**
   - Kolom `Salary` masih bertipe *object* (string) dengan format beragam (contoh: '48K+'), sehingga perlu pembersihan sebelum dianalisis secara numerik.

### Tahap 3: Cleaning Data
Pada tahap ini, kita akan memperbaiki masalah yang ditemukan pada tahap assessment.

In [41]:
# 1. Menghapus baris duplikat
df_clean = df.drop_duplicates().copy()

# 2. Membersihkan nama kolom dari spasi tersembunyi
df_clean.columns = df_clean.columns.str.strip()

# 3. Menghapus baris dengan missing values pada kolom utama
df_clean.dropna(subset=['Company', 'Job Title', 'Location', 'Job Type'], inplace=True)

# 4. Membersihkan kolom Salary ke numerik
def clean_salary_numeric(val):
    if pd.isna(val): return None
    # Menghapus simbol mata uang, spasi, plus, dan asterisk
    val = str(val).lower().replace('$', '').replace(' ', '').replace('+', '').replace('*', '')
    if 'k' in val:
        try:
            return float(val.replace('k', '')) * 1000
        except:
            return None
    try:
        return float(val)
    except:
        return None

df_clean['Salary_Numeric'] = df_clean['Salary'].apply(clean_salary_numeric)

# 5. Menangani 'Semantic Emptiness' pada kolom Facilities
df_clean.loc[df_clean['Facilities'].str.match(r'^[\s,]*$', na=False), 'Facilities'] = "None Specified"

# 6. Menangani kontradiksi logis
df_clean = df_clean[~((df_clean['Job Title'].str.contains('Senior', case=False, na=False)) &
                      (df_clean['Experience level'] == 'Entry-level'))]

print(f"Data Cleaning Selesai!")
print(f"Jumlah baris awal: {len(df)}")
print(f"Jumlah baris setelah cleaning: {len(df_clean)}")
display(df_clean.head())

Data Cleaning Selesai!
Jumlah baris awal: 3198
Jumlah baris setelah cleaning: 2989


,Company,Job Title,Location,Job Type,Experience level,Salary,Requirment of the company,Facilities,Salary_Numeric
0,SGS,Clinical Data Analyst,"Richardson, TX, United States",Full Time,Entry-level,48K+ *,"Computer Science,Data quality,Genetics,Mathema...",None Specified,48000.0
1,Ocorian,AML/CFT & Data Analyst,"Ebène, Mauritius",Full Time,Entry-level,48K+ *,"Agile,Data management,Finance,Security,,",None Specified,48000.0
2,Cricut,Machine Learning Engineer,"South Jordan, UT, United States",Full Time,NaN,90K+ *,"Agile,Architecture,AWS,Computer Science,Comput...","Career development,,,,",90000.0
3,Bosch Group,Application Developer & Data Analyst,"Nonantola, Italy",Full Time,Entry-level,48K+ *,"Engineering,Industrial,Oracle,Power BI,R,R&D",None Specified,48000.0
4,Publicis Groupe,Data Engineer Full time (Public Sector) USA,"Arlington, VA, United States",Full Time,Mid-level,108K+,"AWS,Azure,Computer Science,Consulting,Dataflow...","Flex hours,Flex vacation,Parental leave,Unlimi...",108000.0


In [42]:
# Verifikasi akhir apakah masih ada missing values pada df_clean
print("Jumlah Missing Values Setelah Cleansing:")
print(df_clean.isnull().sum())

# Menghapus baris yang masih memiliki NaN (terutama di Experience level dan Salary_Numeric)
df_clean_final = df_clean.dropna()

print(f"\nJumlah baris akhir setelah penghapusan missing values: {len(df_clean_final)}")
display(df_clean_final.info())

Jumlah Missing Values Setelah Cleansing:
Company                        0
Job Title                      0
Location                       0
Job Type                       0
Experience level             227
Salary                       171
Requirment of the company      0
Facilities                     0
Salary_Numeric               191
dtype: int64

Jumlah baris akhir setelah penghapusan missing values: 2574
<class 'pandas.core.frame.DataFrame'>
Index: 2574 entries, 0 to 3196
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Company                    2574 non-null   object 
 1   Job Title                  2574 non-null   object 
 2   Location                   2574 non-null   object 
 3   Job Type                   2574 non-null   object 
 4   Experience level           2574 non-null   object 
 5   Salary                     2574 non-null   object 
 6   Requirment of the company  2574 no

None

In [43]:
# Kode untuk menghapus satu atau lebih kolom
# Ganti 'nama_kolom' dengan nama kolom yang ingin dihapus
df_clean = df_clean.drop(columns=['Salary','Salary_Numeric','Facilities','Location'], errors='ignore')

# Menampilkan kolom yang tersisa
print("Kolom yang tersedia sekarang:")
print(df_clean.columns.tolist())

Kolom yang tersedia sekarang:
['Company', 'Job Title', 'Job Type', 'Experience level', 'Requirment of the company']


In [44]:
import pandas as pd

# Melakukan standarisasi nama kolom: lowercase dan replace space dengan underscore
df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')

# Jika df_clean_final juga ingin digunakan, lakukan hal yang sama
df_clean_final.columns = df_clean_final.columns.str.lower().str.replace(' ', '_')

print("Standarisasi nama kolom berhasil dilakukan!")
print("Nama kolom sekarang:")
print(df_clean.columns.tolist())

display(df_clean.head())

Standarisasi nama kolom berhasil dilakukan!
Nama kolom sekarang:
['company', 'job_title', 'job_type', 'experience_level', 'requirment_of_the_company']


,company,job_title,job_type,experience_level,requirment_of_the_company
0,SGS,Clinical Data Analyst,Full Time,Entry-level,"Computer Science,Data quality,Genetics,Mathema..."
1,Ocorian,AML/CFT & Data Analyst,Full Time,Entry-level,"Agile,Data management,Finance,Security,,"
2,Cricut,Machine Learning Engineer,Full Time,NaN,"Agile,Architecture,AWS,Computer Science,Comput..."
3,Bosch Group,Application Developer & Data Analyst,Full Time,Entry-level,"Engineering,Industrial,Oracle,Power BI,R,R&D"
4,Publicis Groupe,Data Engineer Full time (Public Sector) USA,Full Time,Mid-level,"AWS,Azure,Computer Science,Consulting,Dataflow..."


In [45]:
# Mengubah nama kolom 'requirment_of_the_company' menjadi 'skills'
df_clean = df_clean.rename(columns={'requirment_of_the_company': 'skills'})
# Lakukan hal yang sama pada df_clean_final agar konsisten
df_clean_final = df_clean_final.rename(columns={'requirment_of_the_company': 'skills'})

print("Kolom berhasil diubah menjadi 'skills'!")
print(df_clean.columns.tolist())
display(df_clean.head())

Kolom berhasil diubah menjadi 'skills'!
['company', 'job_title', 'job_type', 'experience_level', 'skills']


,company,job_title,job_type,experience_level,skills
0,SGS,Clinical Data Analyst,Full Time,Entry-level,"Computer Science,Data quality,Genetics,Mathema..."
1,Ocorian,AML/CFT & Data Analyst,Full Time,Entry-level,"Agile,Data management,Finance,Security,,"
2,Cricut,Machine Learning Engineer,Full Time,NaN,"Agile,Architecture,AWS,Computer Science,Comput..."
3,Bosch Group,Application Developer & Data Analyst,Full Time,Entry-level,"Engineering,Industrial,Oracle,Power BI,R,R&D"
4,Publicis Groupe,Data Engineer Full time (Public Sector) USA,Full Time,Mid-level,"AWS,Azure,Computer Science,Consulting,Dataflow..."


In [46]:
df_clean['job_title'].unique()

array(['Clinical Data Analyst', 'AML/CFT & Data Analyst',
       'Machine Learning Engineer', ...,
       'Application Integration Engineer, Computer Vision Program',
       'Senior Software Engineer, Machine Learning - Ads Intelligence',
       'Data Scientist - New College Graduate'], dtype=object)

In [47]:
# 3. Tampilkan hasil distribusi
distribusi = df_clean['job_title'].value_counts()
display(distribusi)

,count
job_title,
Data Engineer,104
Data Scientist,82
Data Analyst,77
Senior Data Engineer,63
Machine Learning Engineer,47
...,...
"Data Scientist | Insights (f/m/d) - GER, UK, NL, PL",1
Senior Data Analyst - Sales,1
Sr Software QA Engineer-Machine Learning QE,1


In [48]:
df_clean['experience_level'].fillna('No Experience', inplace=True)
print("Missing values in 'experience_level' filled with 'No Experience'.")
df_clean.isna().sum()

Missing values in 'experience_level' filled with 'No Experience'.


/tmp/ipykernel_3041/2692574682.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['experience_level'].fillna('No Experience', inplace=True)


,0
company,0
job_title,0
job_type,0
experience_level,0
skills,0


In [49]:
print(f"Jumlah duplikat sebelum penghapusan: {df_clean.duplicated().sum()}")
df_clean.drop_duplicates(inplace=True)
print(f"Jumlah duplikat setelah penghapusan: {df_clean.duplicated().sum()}")

Jumlah duplikat sebelum penghapusan: 86
Jumlah duplikat setelah penghapusan: 0


In [50]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2903 entries, 0 to 3196
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   company           2903 non-null   object
 1   job_title         2903 non-null   object
 2   job_type          2903 non-null   object
 3   experience_level  2903 non-null   object
 4   skills            2903 non-null   object
dtypes: object(5)
memory usage: 136.1+ KB


### Pembersihan Simbol pada Kolom 'job_title' dan 'skills'

Untuk memastikan konsistensi dan kemudahan analisis, kita akan menghapus atau mengganti simbol-simbol yang tidak perlu dari kolom `job_title` dan `skills`. Ini akan membantu dalam proses standarisasi lebih lanjut.

Berikut adalah contoh data sebelum pembersihan:

In [51]:
display(df_clean[['job_title', 'skills']].head())

,job_title,skills
0,Clinical Data Analyst,"Computer Science,Data quality,Genetics,Mathema..."
1,AML/CFT & Data Analyst,"Agile,Data management,Finance,Security,,"
2,Machine Learning Engineer,"Agile,Architecture,AWS,Computer Science,Comput..."
3,Application Developer & Data Analyst,"Engineering,Industrial,Oracle,Power BI,R,R&D"
4,Data Engineer Full time (Public Sector) USA,"AWS,Azure,Computer Science,Consulting,Dataflow..."


In [52]:
import re
import pandas as pd

def clean_text_for_job_title(text):
    if pd.isna(text) or not isinstance(text, str):
        return text
    text = text.lower()
    # Hapus karakter yang bukan alfanumerik, spasi, atau karakter spesifik yang diizinkan
    # Diizinkan: alfanumerik, spasi, /, -, &, (, ), koma
    text = re.sub(r'[^a-z0-9\s/&,()-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_for_skills(text):
    if pd.isna(text) or not isinstance(text, str):
        return text
    text = text.lower()
    # Diizinkan: alfanumerik, spasi, koma, plus (+), hash (#), titik (.), garis miring (/), hyphen (-)
    text = re.sub(r'[^a-z0-9\s,+#./-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Terapkan fungsi pembersihan langsung ke kolom asli
df_clean['job_title'] = df_clean['job_title'].apply(clean_text_for_job_title)
df_clean['skills'] = df_clean['skills'].apply(clean_text_for_skills)

print("\nPembersihan simbol pada kolom 'job_title' dan 'skills' selesai.")


Pembersihan simbol pada kolom 'job_title' dan 'skills' selesai.


Berikut adalah contoh data sesudah pembersihan:

In [53]:
display(df_clean[['job_title', 'skills']].head())

,job_title,skills
0,clinical data analyst,"computer science,data quality,genetics,mathema..."
1,aml/cft & data analyst,"agile,data management,finance,security,,"
2,machine learning engineer,"agile,architecture,aws,computer science,comput..."
3,application developer & data analyst,"engineering,industrial,oracle,power bi,r,rd"
4,data engineer full time (public sector) usa,"aws,azure,computer science,consulting,dataflow..."


### Ekstraksi dan Analisis Keterampilan (Skills Extraction and Analysis)

Sekarang, kita akan fokus pada kolom 'skills' untuk mengekstrak dan menganalisis keterampilan individual yang disebutkan dalam setiap lowongan pekerjaan. Langkah-langkah ini akan membantu kita mengidentifikasi keterampilan yang paling sering dicari.

In [54]:
import numpy as np

def extract_and_clean_skills(skills_text):
    if pd.isna(skills_text) or not isinstance(skills_text, str):
        return []
    # Membagi string skills berdasarkan berbagai delimiter yang mungkin
    # Contoh delimiter: koma, titik koma, garis miring, 'and'
    skills_list = re.split(r'[,;/]| and ', skills_text.lower())

    cleaned_skills = []
    for skill in skills_list:
        skill = skill.strip()
        # Hapus karakter non-alfanumerik kecuali spasi, hash, plus, titik
        skill = re.sub(r'[^a-z0-9\s#+.]', '', skill)
        # Ganti multiple spaces dengan single space
        skill = re.sub(r'\s+', ' ', skill).strip()
        if skill and len(skill) > 0:  # Pastikan skill tidak kosong atau hanya 1 karakter
            cleaned_skills.append(skill)
    return list(set(cleaned_skills)) # Hapus duplikat dalam satu entri skills

# Terapkan fungsi untuk mengekstrak dan membersihkan skills
df_clean['skills'] = df_clean['skills'].apply(extract_and_clean_skills)

print("Kolom 'skills' telah diekstrak dan dibersihkan menjadi daftar individual.")
display(df_clean[['skills']].head(20))

Kolom 'skills' telah diekstrak dan dibersihkan menjadi daftar individual.


,skills
0,"[data quality, mathematics, genetics, sas, com..."
1,"[data management, security, finance, agile]"
2,"[computer vision, architecture, deep learning,..."
3,"[oracle, industrial, r, engineering, power bi,..."
4,"[azure, consulting, dataflow, data pipelines, ..."
5,"[numpy, machine learning, industrial, deep lea..."
6,"[excel, security, data quality, banking]"
7,"[excel, genetics, business intelligence]"
8,"[machine learning, matlab, engineering, mathem..."
9,"[azure, apis, aws, agile, big data, computer s..."


In [55]:
# Explode the dataframe to have one skill per row
df_skills_exploded = df_clean.explode('skills')

# Hitung frekuensi masing-masing skill
skill_counts = df_skills_exploded['skills'].value_counts()

print("10 Keterampilan Teratas yang Paling Sering Dicari:")
display(skill_counts.head(10))

10 Keterampilan Teratas yang Paling Sering Dicari:


,count
skills,
computer science,1056
engineering,984
aws,786
architecture,705
big data,541
agile,539
data analysis,535
azure,490
machine learning,474


### Pemeriksaan Akhir Kebersihan Data

Setelah serangkaian proses pembersihan dan standarisasi, mari kita lakukan pemeriksaan akhir pada DataFrame `df_clean` untuk memastikan bahwa semua langkah telah berhasil dan data siap untuk analisis lebih lanjut.

In [56]:
print("Lima baris pertama dari DataFrame `df_clean` setelah pembersihan:")
display(df_clean.head())

Lima baris pertama dari DataFrame `df_clean` setelah pembersihan:


,company,job_title,job_type,experience_level,skills
0,SGS,clinical data analyst,Full Time,Entry-level,"[data quality, mathematics, genetics, sas, com..."
1,Ocorian,aml/cft & data analyst,Full Time,Entry-level,"[data management, security, finance, agile]"
2,Cricut,machine learning engineer,Full Time,No Experience,"[computer vision, architecture, deep learning,..."
3,Bosch Group,application developer & data analyst,Full Time,Entry-level,"[oracle, industrial, r, engineering, power bi,..."
4,Publicis Groupe,data engineer full time (public sector) usa,Full Time,Mid-level,"[azure, consulting, dataflow, data pipelines, ..."


In [57]:
print("Informasi umum DataFrame `df_clean`:")
display(df_clean.info())

Informasi umum DataFrame `df_clean`:
<class 'pandas.core.frame.DataFrame'>
Index: 2903 entries, 0 to 3196
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   company           2903 non-null   object
 1   job_title         2903 non-null   object
 2   job_type          2903 non-null   object
 3   experience_level  2903 non-null   object
 4   skills            2903 non-null   object
dtypes: object(5)
memory usage: 136.1+ KB


None

In [58]:
print("Jumlah nilai yang hilang di setiap kolom dari `df_clean`:")
display(df_clean.isna().sum())

Jumlah nilai yang hilang di setiap kolom dari `df_clean`:


,0
company,0
job_title,0
job_type,0
experience_level,0
skills,0


In [59]:
# Ekspor df_clean_final ke file CSV
output_file_path = 'cleaned_data_job.csv'
df_clean.to_csv(output_file_path, index=False)

print(f"DataFrame 'dataset' berhasil diekspor ke '{output_file_path}'")

DataFrame 'dataset' berhasil diekspor ke 'cleaned_data_job.csv'


In [60]:
new =pd.read_csv('cleaned_data_job.csv')

In [61]:
new[new.duplicated(keep=False)]

,company,job_title,job_type,experience_level,skills
222,84.51°,senior data engineer (p3949),Full Time,Senior-level,"['azure', 'business analytics', 'cassandra', '..."
821,Visa,financial data analyst,Full Time,No Experience,"['excel', 'data analytics', 'power bi', 'finan..."
1522,Visa,financial data analyst,Full Time,No Experience,"['excel', 'data analytics', 'power bi', 'finan..."
1856,Version 1,data analyst financial services talent pipeline,Full Time,Senior-level,"['consulting', 'etl', 'engineering', 'aws', 'h..."
1999,Version 1,data analyst financial services talent pipeline,Full Time,Senior-level,"['consulting', 'etl', 'engineering', 'aws', 'h..."
2143,84.51°,senior data engineer (p3949),Full Time,Senior-level,"['azure', 'business analytics', 'cassandra', '..."


In [62]:
print(f"Jumlah duplikat sebelum penghapusan: {new.duplicated().sum()}")
new.drop_duplicates(inplace=True)
print(f"Jumlah duplikat setelah penghapusan: {new.duplicated().sum()}")

Jumlah duplikat sebelum penghapusan: 3
Jumlah duplikat setelah penghapusan: 0


In [63]:
new.isnull().sum()

,0
company,0
job_title,0
job_type,0
experience_level,0
skills,0


In [64]:
# Ekspor df_clean_final ke file CSV
output_file_path = 'cleaned_dataset_job.csv'
df_clean.to_csv(output_file_path, index=False)

print(f"DataFrame 'dataset' berhasil diekspor ke '{output_file_path}'")

DataFrame 'dataset' berhasil diekspor ke 'cleaned_dataset_job.csv'


In [65]:
new

,company,job_title,job_type,experience_level,skills
0,SGS,clinical data analyst,Full Time,Entry-level,"['data quality', 'mathematics', 'genetics', 's..."
1,Ocorian,aml/cft & data analyst,Full Time,Entry-level,"['data management', 'security', 'finance', 'ag..."
2,Cricut,machine learning engineer,Full Time,No Experience,"['computer vision', 'architecture', 'deep lear..."
3,Bosch Group,application developer & data analyst,Full Time,Entry-level,"['oracle', 'industrial', 'r', 'engineering', '..."
4,Publicis Groupe,data engineer full time (public sector) usa,Full Time,Mid-level,"['azure', 'consulting', 'dataflow', 'data pipe..."
...,...,...,...,...,...
2898,CCRi,"application integration engineer, computer vis...",Full Time,Mid-level,"['azure', 'apis', 'angular', 'architecture', '..."
2899,Publicis Groupe,"associate director, data science",Full Time,Mid-level,"['data analysis', 'bayesian', 'clustering', 'm..."
2900,DoorDash,"senior software engineer, machine learning - a...",Full Time,Senior-level,"['feature engineering', 'excel', 'data analysi..."
2901,Western Digital,data scientist - new college graduate,Full Time,Entry-level,"['apis', 'clustering', 'deep learning', 'docke..."


In [66]:
# 3. Tampilkan hasil distribusi
distribusi = df_clean['job_title'].value_counts()
display(distribusi)

,count
job_title,
data engineer,106
data scientist,81
data analyst,72
senior data engineer,62
machine learning engineer,46
...,...
staff data scientist - nlp,1
data science intern - large language model,1
data architect (optional relocation to montenegro),1
